# Basics &mdash; Equivalence Relations and Partitioning

**Concept 5 of the Basics decomposition:** *Equivalence Relations and Partitioning*

Reflexive + symmetric + transitive; every such relation partitions the set into disjoint classes.

---

*Run on Colab:* [![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ganeshutah/Jove/blob/master/Basics/Concept-Equivalence-Relations/Concept-Equivalence-Relations.ipynb)
*Or run locally from inside a Jove checkout.*

## 0. Setup

In [ ]:
# Run me first.  Works on Colab and on a local Jove checkout.
import os, subprocess, sys

def _git(*a):
    r = subprocess.run(('git',) + a, capture_output=True, text=True)
    return r.stdout.strip() if r.returncode == 0 else ''

REPO = 'https://github.com/ganeshutah/Jove'
try:                       # ---- Colab: clone once, pull thereafter ----
    import google.colab
    was = _git('-C', 'Jove', 'rev-parse', '--short', 'HEAD') if os.path.isdir('Jove') else ''
    if os.path.isdir('Jove') and not was:
        print('Jove: WARNING ./Jove exists but is not a git checkout -- left as is')
    elif was:
        _git('-C', 'Jove', 'pull', '-q', '--ff-only')
        now = _git('-C', 'Jove', 'rev-parse', '--short', 'HEAD')
        if now and now != was:
            print('Jove: PULLED  %s -> %s' % (was, now))
            print(_git('-C', 'Jove', 'log', '--oneline', was + '..' + now))
        else:
            print('Jove: PULLED  already current at %s' % (now or was))
    else:
        _git('clone', '-q', REPO, 'Jove')
        print('Jove: CLONED  at %s' % (_git('-C', 'Jove', 'rev-parse',
                                             '--short', 'HEAD') or '?'))
    JOVE = 'Jove'
except ImportError:        # ---- local: the checkout above Basics/ ----
    JOVE = next((p for p in ('../..', '../../..', '..', '.')
                 if os.path.isdir(os.path.join(p, 'jove'))), '../..')
    print('Jove: LOCAL   checkout at %s'
          % (_git('-C', JOVE, 'rev-parse', '--short', 'HEAD') or '?'))
sys.path.insert(0, JOVE)

# A session can already hold an OLDER jove in sys.modules.  The pull above
# updates the files on disk, but `import` would hand back the cached module --
# so a fixed library still behaves like the broken one.  Drop them first.
for _m in [k for k in list(sys.modules) if k == 'jove' or k.startswith('jove.')]:
    del sys.modules[_m]

from jove.Def_md2mc      import *
from jove.DotBashers     import *
from jove.Def_DFA        import *
from jove.LangDef        import *
from jove.AnimateDFA     import *

import jove; print('Jove loaded from', list(jove.__path__)[0])
import jove.AnimateDFA as _a; print('animation toolbar:',
      'ready' if hasattr(_a.AnimateDFA, '_ipython_display_')
      else 'STALE -- restart the runtime, then re-run')

## 1. The idea


A binary relation $\equiv$ over a nonempty $S$ is an **equivalence relation** when it
is

* **reflexive** &mdash; $a \equiv a$,
* **symmetric** &mdash; $a \equiv b \Rightarrow b \equiv a$,
* **transitive** &mdash; $a \equiv b \wedge b \equiv c \Rightarrow a \equiv c$.

Every equivalence relation **partitions** $S$ into **pairwise disjoint** classes whose
union is all of $S$.

The book's running example: over $Nat$, $a \equiv b$ iff $a \bmod 3 = b \bmod 3$. That
gives three classes &mdash; and they are exactly the three states of the divisible-by-3
DFA of Chapter 5.

**Transitivity is the clause that does the work later.** It is why overlapping
equivalent state-pairs must coalesce in DFA minimization (Chapter 6, Concept 9).

## 2. Definitions

### Checking the three properties

In [ ]:
def is_equivalence(S, rel):
    refl  = all(rel(a, a) for a in S)
    symm  = all(rel(a, b) == rel(b, a) for a in S for b in S)
    trans = all((not (rel(a, b) and rel(b, c))) or rel(a, c)
                for a in S for b in S for c in S)
    return refl, symm, trans, (refl and symm and trans)

def classes_of(S, rel):
    out = []
    for a in sorted(S):
        for cl in out:
            if rel(a, cl[0]): cl.append(a); break
        else:
            out.append([a])
    return out

### Candidate relations

In [ ]:
S = set(range(12))
REL = {
 'a mod 3 == b mod 3' : lambda a, b: a % 3 == b % 3,
 'same parity'        : lambda a, b: a % 2 == b % 2,
 'a <= b'             : lambda a, b: a <= b,          # not symmetric
 '|a - b| <= 1'       : lambda a, b: abs(a - b) <= 1, # not transitive
 'a != b'             : lambda a, b: a != b,          # not reflexive
}

<!-- nav-strip -->

---

&larr;&nbsp;[Basics&nbsp;4.&nbsp;Complement, Relative to a Universe](https://colab.research.google.com/github/ganeshutah/Jove/blob/master/Basics/Concept-Complement-Relative-To-Universe/Concept-Complement-Relative-To-Universe.ipynb) &nbsp;&middot;&nbsp; [**Basics** index](https://github.com/ganeshutah/Jove/blob/master/Basics/README.md) &nbsp;&middot;&nbsp; [Basics&nbsp;6.&nbsp;Logical Connectives and Predicates](https://colab.research.google.com/github/ganeshutah/Jove/blob/master/Basics/Concept-Logical-Connectives/Concept-Logical-Connectives.ipynb)&nbsp;&rarr;

---

## 3. Tests

Which of the five are equivalence relations?

In [ ]:
print("%-22s %-8s %-8s %-8s %s" % ("relation", "refl", "symm", "trans", "equivalence?"))
for name, r in REL.items():
    refl, symm, trans, ok = is_equivalence(S, r)
    print("%-22s %-8s %-8s %-8s %s" % (name, refl, symm, trans, ok))
assert is_equivalence(S, REL['a mod 3 == b mod 3'])[3]
assert not is_equivalence(S, REL['a <= b'])[3]

The failures, with explicit witnesses.

In [ ]:
r = REL['a <= b']
print("  'a <= b' not symmetric :  1<=2 is %s but 2<=1 is %s" % (r(1, 2), r(2, 1)))
r = REL['|a - b| <= 1']
print("  '|a-b|<=1' not transitive: 1~2 %s, 2~3 %s, but 1~3 %s"
      % (r(1, 2), r(2, 3), r(1, 3)))
r = REL['a != b']
print("  'a != b' not reflexive  :  1~1 is %s" % r(1, 1))

An equivalence relation **partitions** the set.

In [ ]:
cls = classes_of(S, REL['a mod 3 == b mod 3'])
for cl in cls: print("   class:", cl)
flat = [x for cl in cls for x in cl]
assert sorted(flat) == sorted(S)                     # union is all of S
assert len(flat) == len(set(flat))                   # pairwise disjoint
print("\nunion is all of S, and the classes are pairwise disjoint")

**And those three classes are the three states of the mod-3 DFA.**

In [ ]:
L3Z = md2mc('''DFA
IF : 0 -> S1
IF : 1 -> IF
S1 : 0 -> S2
S1 : 1 -> S1
S2 : 0 -> IF
S2 : 1 -> S2
''')
print("states :", sorted(L3Z["Q"]), " -- one per residue class")
assert len(L3Z["Q"]) == len(cls) == 3
tag = {'IF': 0, 'S1': 1, 'S2': 2}
for n in range(10):
    w = '0' * n
    assert tag[run_dfa(L3Z, w)] == n % 3
print("state reached by 0^n is the class of n mod 3, for n = 0..9")

**Transitivity** is what forces class merging in DFA minimization.

In [ ]:
pairs = [('F1', 'F2'), ('F2', 'F3')]
print("equivalent pairs found by minimization :", pairs)
print("transitivity forces F1 ~ F3, so the class is {F1, F2, F3}")
print()
print("That coalescing step is Chapter 6, Concept 9 -- and it is legal only")
print("because indistinguishability is transitive.")

## 4. Animation

The three residue classes, as the three states of a machine.

In [ ]:
from jove.AnimateDFA import *
AnimateDFA(L3Z, FuseEdges=True)

## 5. Exercises


1. Is "same length" an equivalence relation on $\Sigma^*$? How many classes?
2. Show that indistinguishability of DFA states is an equivalence relation.
3. How many equivalence relations are there on a 3-element set?

In [ ]:
# Your work for the exercises above.

## 6. Where next

In [ ]:
# Previous / next, and a search box for every concept.
# Type a chapter (Chapter7, ch7, NFA) or words from a title (pumping).
#
# Following a link opens a NEW Colab runtime. To pull another concept's
# definitions into THIS session instead:
#     load_here('Chapter7-NFA/Concept-...')
import os, sys
try:                       # usually already done by the Setup cell
    import jove
except ModuleNotFoundError:
    _p = next((p for p in ('Jove', '../..', '../../..', '..', '.')
               if os.path.isdir(os.path.join(p, 'jove'))), None)
    if _p:
        sys.path.insert(0, _p)
try:
    from jove.Nav import nav, load_here
    nav(here='Basics/Concept-Equivalence-Relations')
except ModuleNotFoundError:
    print('Jove is not on the path yet.')
    print('Run the Setup cell at the top of this notebook, then re-run this one.')